In [46]:
import torch
import glob
import os
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from pathlib import Path

from utils import functions as uf
from utils.model import DynamicMLP
from sklearn.model_selection import train_test_split


In [47]:
folder_path = './data/'
device = torch.device('cpu')

# Caricamento di tutti i file CSV presenti nella cartella
csv_files = glob.glob(os.path.join(folder_path, '*c000.csv'))
df_all = pd.concat((pd.read_csv(file, sep=";") for file in csv_files), ignore_index=True)
    
random_seed = 456
id_col = "user_id"

# Caricamento Forget set
forget_file_path = os.path.join(folder_path, 'forget_data.csv')
forget_df = pd.read_csv(forget_file_path)

# Estrazione ID da dimenticare
forget_ids = set(forget_df[id_col])

# Separazione netta tra Retain Set e Forget Set
retain_df = df_all[~df_all[id_col].isin(forget_ids)].reset_index(drop=True)
forget_df = df_all[df_all[id_col].isin(forget_ids)].reset_index(drop=True)

# Split Retain -> Train (80%) e Temp (20%)
train_df, temp_df = train_test_split(retain_df, test_size=0.20, random_state=random_seed)

# Split Temp -> Val (10%) e Test (10%)
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=random_seed)

print(f"Dataset Sizes -> Train (Retain): {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)} | Forget: {len(forget_df)}")

# Salva gli ID di validazione richiesti per la submission
#val_df[[id_col]].to_csv('validation_ids.csv', index=False)

Dataset Sizes -> Train (Retain): 96558 | Val: 12070 | Test: 12070 | Forget: 9085


In [48]:
# 1. Estrazione delle matrici X e y per tutti i set dal dataframe corretto
X_train_raw, y_train, feature_cols, target_cols = uf.prepare_data(train_df, id_col=id_col, target_prefix='target__')
X_val_raw, y_val, _, _ = uf.prepare_data(val_df, id_col=id_col, target_prefix='target__')
X_test_raw, y_test, _, _ = uf.prepare_data(test_df, id_col=id_col, target_prefix='target__')
X_forget_raw, y_forget, _, _ = uf.prepare_data(forget_df, id_col=id_col, target_prefix='target__')

# 2. Imputazione coerente dei dati (fit su tutto df_all trasformato per coerenza con l'addestramento originale)
X_all_raw, _, _, _ = uf.prepare_data(df_all, id_col=id_col, target_prefix='target__')
imputer = SimpleImputer(strategy='median')
imputer.fit(X_all_raw)

X_train = imputer.transform(X_train_raw).astype(np.float32)
X_val = imputer.transform(X_val_raw).astype(np.float32)
X_test = imputer.transform(X_test_raw).astype(np.float32)
X_forget = imputer.transform(X_forget_raw).astype(np.float32)

# 3. Calcolo pos_weights sul Retain Train Set per la loss
pos_counts = np.sum(y_train, axis=0)
neg_counts = len(y_train) - pos_counts
pos_weights = torch.tensor(neg_counts / (pos_counts + 1e-6), dtype=torch.float32, device=device)
pos_weights = pos_weights.clamp(min=0.1, max=100.0)
print(f"pos_weights: {pos_weights}")

# 4. Caricamento Artifact del modello pre-addestrato
artifact_path = Path('data') / 'model_artifact'
payload = uf.load_pickle(artifact_path)

state_dict = payload['state_dict']
architecture = payload['architecture']
best_params = payload['best_hyperparameters']

print("\n--- Saved Metadata ---")
print("Architecture parameters:", architecture)
print("Best Hyperparameters:", best_params)

model = DynamicMLP(
    input_dim=architecture['input_dim'],
    hidden_layers=architecture['hidden_layers'],
    num_outputs=architecture['num_outputs']
).to(device)

model.load_state_dict(state_dict)
model.eval()

print("\nModel successfully reconstructed and weights loaded.")

DataFrame columns: ['ang_m_datediff_att_linea', 'ang_m_flag_mnp', 'ang_m_datediff_cessazione', 'ang_m_datediff_scad_sim', 'ang_m_cns_cre_res_per', 'ang_m_anz_linea', 'ang_m_datediff_mnp_in', 'ang_m_datediff_cam_prof', 'ang_m_datediff_mnp_out', 'ohe_ang_m_stato_linea_a', 'ohe_ang_m_stato_linea_c', 'ohe_ang_m_stato_linea_n', 'ohe_ang_m_stato_linea_s', 'ohe_ang_m_tipo_linea_fwa', 'ohe_ang_m_tipo_linea_pp', 'ohe_ang_m_tipo_linea_xt', 'ohe_ang_m_area_territoriale_altro_nd', 'ohe_ang_m_area_territoriale_c', 'ohe_ang_m_area_territoriale_ne', 'ohe_ang_m_area_territoriale_no', 'ohe_ang_m_area_territoriale_s', 'ohe_ang_m_pdv_canale_des_top_4_altro', 'ohe_ang_m_pdv_canale_des_top_4_grande_distribuzione', 'ohe_ang_m_pdv_canale_des_top_4_null', 'ohe_ang_m_pdv_canale_des_top_4_one_team', 'ohe_ang_m_pdv_canale_des_top_4_retail', 'ohe_ang_m_pdv_canale_des_top_4_tele_sales', 'ohe_ang_m_mnp_olo_prov_infrastrutturati', 'ohe_ang_m_mnp_olo_prov_mvno', 'ohe_ang_m_mnp_olo_prov_nd', 'ohe_ang_m_mnp_olo_dest_in

In [49]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import copy

# 1. Creiamo i DataLoader per Forget e Retain
batch_size = 256

# Prepareshing dei PyTorch Tensor
train_dataset = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.float32))
forget_dataset = TensorDataset(torch.tensor(X_forget, dtype=torch.float32), torch.tensor(y_forget, dtype=torch.float32))

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
forget_loader = DataLoader(forget_dataset, batch_size=batch_size, shuffle=True)

# 2. Definiamo la Loss Function con i pos_weights calcolati in precedenza
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights)

# Gradeint ascent method

In [11]:
import copy
import time
import torch
import torch.nn as nn
import torch.optim as optim

start_time = time.time()

# ---------------------------------------------------------
# FASE 1: UNLEARNING (COMBINED ASCENT + DESCENT)
# ---------------------------------------------------------
print("\n--- FASE 1: Combined Unlearning (Gradient Ascent + Descent) ---")

unlearned_model = copy.deepcopy(model)
unlearned_model.train()

criterion = nn.BCEWithLogitsLoss().to(device)
optimizer = optim.Adam(unlearned_model.parameters(), lr=1e-4)

alpha = 1.0  # Peso Ascent (Forget Set)
beta = 0   # Peso Descent (Retain Set)
epochs_unlearn = 1

train_iter = iter(train_loader)

for epoch in range(epochs_unlearn):
    total_loss = 0.0
    for x_f, y_f in forget_loader:
        x_f, y_f = x_f.to(device), y_f.to(device)
        
        try:
            x_r, y_r = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader)
            x_r, y_r = next(train_iter)
            
        x_r, y_r = x_r.to(device), y_r.to(device)
        
        optimizer.zero_grad()
        
        out_f = unlearned_model(x_f)
        loss_forget = criterion(out_f, y_f)
        
        out_r = unlearned_model(x_r)
        loss_retain = criterion(out_r, y_r)
        
        combined_loss = - alpha * loss_forget + beta * loss_retain
        combined_loss.backward()
        optimizer.step()
        
        total_loss += combined_loss.item()
        
    print(f"Unlearning Epoch {epoch+1}/{epochs_unlearn} - Loss combinata: {total_loss / len(forget_loader):.4f}")


# ---------------------------------------------------------
# FASE 2: FINE-TUNING VIA UTILS FUNCTION
# ---------------------------------------------------------
unlearned_model, best_val_p10 = uf.run_finetune_with_p10(
    model=unlearned_model,
    train_loader=train_loader,
    X_val=X_val,
    y_val=y_val,
    device=device,
    uf_module=uf,
    epochs=2,
    lr=3e-4
)


# ---------------------------------------------------------
# FASE 3: ESPORTAZIONE SUBMISSION BLINDATA
# ---------------------------------------------------------
submission_path = uf.export_submission(
    model=unlearned_model,
    val_df=val_df,
    id_col=id_col,
    architecture=architecture,
    best_params=best_params,
    version="V34",
    group_name="NoWINDtoday",
    payload=payload,
    start_time=start_time
)


--- FASE 1: Combined Unlearning (Gradient Ascent + Descent) ---
Unlearning Epoch 1/1 - Loss combinata: -0.4306

--- Inizio Fine-Tuning sul Retain Set (2 epoche) ---
Epoch 1/2 | Retain Loss: 0.1370 | Train P@10: 0.0345 | Val P@10: 0.0396
Epoch 2/2 | Retain Loss: 0.0817 | Train P@10: 0.0407 | Val P@10: 0.0409
✅ Caricati i pesi dell'epoca con la migliore Val Precision@10: 0.0409
--- Check Dtypes dello State Dict ---
Tensori Float32: 14 | Tensori Float64: 0
✅ Perfetto: NESSUN tensore a 64-bit presente! Tutto in Float32.

✅ Cartella 'submissions\NoWINDtoday_V34' creata con successo e pronta per l'invio!


# Fischer

In [59]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import copy
# =========================================================
# FASE 1: CALCOLO DELL'INFORMAZIONE DI FISHER SUL RETAIN SET
# =========================================================
print("\n--- Calcolo dell'Informazione di Fisher sul Retain Set ---")

# Utilizziamo una copia del modello originale già caricato
fisher_model = copy.deepcopy(model)
fisher_model.eval()

# Inizializzazione contenitore per il quadrato dei gradienti (Fisher Matrix Diagonale)
fisher_diag = {
    name: torch.zeros_like(param) 
    for name, param in fisher_model.named_parameters() 
    if param.requires_grad
}

num_samples = 0

# Accumulo dei gradienti al quadrato sul train_loader (Retain Set)
for x_b, y_b in train_loader:
    x_b, y_b = x_b.to(device), y_b.to(device)
    batch_dim = x_b.size(0)
    num_samples += batch_dim
    
    fisher_model.zero_grad()
    out = fisher_model(x_b)
    loss = criterion(out, y_b)
    loss.backward()
    
    with torch.no_grad():
        for name, param in fisher_model.named_parameters():
            if param.requires_grad and param.grad is not None:
                fisher_diag[name] += (param.grad ** 2) * batch_dim

# Media dell'Informazione di Fisher
for name in fisher_diag:
    fisher_diag[name] /= num_samples


# =========================================================
# FASE 2: INIEZIONE DI RUMORE DI FISHER (UNLEARNING)
# =========================================================
print("\n--- Iniezione di Rumore di Fisher (Selective Amnesia) ---")

alpha_fisher = 1e-5  # Scala del rumore conservativa
damping = 1e-5       # Damping elevato per stabilità (evita divisioni per zero)

with torch.no_grad():
    for name, param in fisher_model.named_parameters():
        if param.requires_grad:
            # Protezione: Applichiamo il rumore SOLO ai pesi (weight) e NON a BatchNorm o Bias
            if 'weight' in name and 'bn' not in name and 'batch' not in name:
                variance = alpha_fisher / torch.sqrt(fisher_diag[name] + damping)
                std_dev = torch.sqrt(variance)
                
                noise = torch.randn_like(param) * std_dev
                param.add_(noise)

print("Fisher Forgetting applicato con successo!")


# =========================================================
# FASE 3: FINE-TUNING DI RICALIBRAZIONE SUL RETAIN SET
# =========================================================
fisher_model, best_val_p10 = uf.run_finetune_with_p10(
    model=fisher_model,
    train_loader=train_loader,
    X_val=X_val,
    y_val=y_val,
    device=device,
    uf_module=uf,
    epochs=2,
    lr=0.0008
)


# =========================================================
# FASE 4: CREAZIONE SUBMISSION BLINDATA
# =========================================================

submission_path = uf.export_submission(
    model=fisher_model,
    val_df=val_df,
    id_col=id_col,
    architecture=architecture,
    best_params=best_params,
    payload=payload,
    version="final",
    group_name="NoWINDtoday"
)


--- Calcolo dell'Informazione di Fisher sul Retain Set ---

--- Iniezione di Rumore di Fisher (Selective Amnesia) ---
Fisher Forgetting applicato con successo!

--- Inizio Fine-Tuning sul Retain Set (2 epoche) ---
Epoch 1/2 | Retain Loss: 0.0983 | Train P@10: 0.0385 | Val P@10: 0.0410
Epoch 2/2 | Retain Loss: 0.0804 | Train P@10: 0.0416 | Val P@10: 0.0412
✅ Caricati i pesi dell'epoca con la migliore Val Precision@10: 0.0412
--- Check Dtypes dello State Dict ---
Tensori Float32: 14 | Tensori Float64: 0
✅ Perfetto: NESSUN tensore a 64-bit presente! Tutto in Float32.

✅ Cartella 'submissions\NoWINDtoday_final' creata con successo e pronta per l'invio!


In [58]:
import torch

def compare_state_dicts(state_dict_1, state_dict_2):
    keys_1 = set(state_dict_1.keys())
    keys_2 = set(state_dict_2.keys())
    
    # 1. Verifica presenza chiavi
    missing_in_2 = keys_1 - keys_2
    missing_in_1 = keys_2 - keys_1
    
    if missing_in_2 or missing_in_1:
        print("❌ Discrepanza nelle chiavi!")
        if missing_in_2:
            print(f"  - Chiavi presenti solo nel primo: {missing_in_2}")
        if missing_in_1:
            print(f"  - Chiavi presenti solo nel secondo: {missing_in_1}")
        return False

    print("✅ Le chiavi dei due modelli combaciano perfettamente.\n")

    # 2. Confronto valore dei tensor
    identical_params = []
    different_params = []

    for key in keys_1:
        # Convertiamo sempre in float per evitare il RuntimeError su interi/Long
        tensor1 = state_dict_1[key].cpu().float()
        tensor2 = state_dict_2[key].cpu().float()
        
        # Verifica se i valori sono uguali
        if torch.equal(tensor1, tensor2):
            identical_params.append(key)
        else:
            diff = torch.abs(tensor1 - tensor2)
            max_diff = torch.max(diff).item()
            mean_diff = torch.mean(diff).item()  # Ora funziona perfettamente su tutti i layer!
            different_params.append((key, max_diff, mean_diff))

    # Stampa i risultati
    print(f"Parametri IDENTICI: {len(identical_params)} / {len(keys_1)}")
    print(f"Parametri MODIFICATI: {len(different_params)} / {len(keys_1)}\n")

    if different_params:
        print("Dettaglio parametri modificati (Layer | Max Diff | Mean Diff):")
        for key, max_d, mean_d in different_params:
            print(f" - {key:<45} | Max: {max_d:.6e} | Mean: {mean_d:.6e}")
    else:
        print("🎉 I due modelli sono al 100% uguali!")

    return len(different_params) == 0

# --- ESECUZIONE ---
are_identical = compare_state_dicts(payload['state_dict'], fisher_model.state_dict())

✅ Le chiavi dei due modelli combaciano perfettamente.

Parametri IDENTICI: 1 / 16
Parametri MODIFICATI: 15 / 16

Dettaglio parametri modificati (Layer | Max Diff | Mean Diff):
 - net.6.bias                                    | Max: 7.435497e-02 | Mean: 4.671959e-02
 - net.3.weight                                  | Max: 2.600885e-01 | Mean: 5.161708e-02
 - net.4.bias                                    | Max: 1.418231e-01 | Mean: 5.632420e-02
 - net.1.num_batches_tracked                     | Max: 3.780000e+02 | Mean: 3.780000e+02
 - net.1.running_var                             | Max: 3.634313e+20 | Mean: 6.875080e+19
 - net.1.weight                                  | Max: 1.330650e-01 | Mean: 4.202931e-02
 - net.4.weight                                  | Max: 1.983500e-01 | Mean: 6.823653e-02
 - net.0.weight                                  | Max: 2.229867e-01 | Mean: 4.502240e-02
 - net.4.num_batches_tracked                     | Max: 3.780000e+02 | Mean: 3.780000e+02
 - net.1.runni

# KL Divergence

In [7]:
# =========================================================
# FASE 1: UNLEARNING TRAMITE KL DIVERGENCE 
# (Prende il posto del ciclo di Fisher!)
# =========================================================
print("\n--- FASE 1: Unlearning con KL Divergence ---")

unlearned_model = copy.deepcopy(model)
unlearned_model.train()

# Model originale congelato (serve da ancora per il Retain)
orig_model = copy.deepcopy(model)
orig_model.eval()
for param in orig_model.parameters():
    param.requires_grad = False

optimizer_unlearn = torch.optim.Adam(unlearned_model.parameters(), lr=1e-4)

# Iperparametri di bilanciamento
alpha = 1.0  # Peso Retain (mantiene la memoria)
beta = 0.3   # Peso Forget (forza l'amnesia verso 0.5)

train_iter = iter(train_loader)

for epoch in range(1): # 1 o 2 epoche bastano
    for x_f, y_f in forget_loader:
        x_f = x_f.to(device)
        
        # Prendi batch dal Retain
        try:
            x_r, _ = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader)
            x_r, _ = next(train_iter)
        x_r = x_r.to(device)
        
        optimizer_unlearn.zero_grad()
        
        # 1. RETAIN: Mantiieni gli output vicini al modello originale
        with torch.no_grad():
            logits_orig_r = orig_model(x_r)
        logits_unlearned_r = unlearned_model(x_r)
        loss_retain = uf.binary_kl_divergence(logits_orig_r, logits_unlearned_r)
        
        # 2. FORGET: Spingi verso la massima incertezza (logit 0.0 -> p = 0.5)
        logits_unlearned_f = unlearned_model(x_f)
        uniform_target_f = torch.zeros_like(logits_unlearned_f)
        loss_forget = uf.binary_kl_divergence(uniform_target_f, logits_unlearned_f)
        
        # Loss combinata
        total_loss = alpha * loss_retain + beta * loss_forget
        total_loss.backward()
        optimizer_unlearn.step()

print("Unlearning con KL completato!")


# =========================================================
# FASE 2: FINE-TUNING FINALE SUL RETAIN SET (Rimane invariata!)
# =========================================================
print("\n--- FASE 2: Fine-Tuning di Ricalibrazione sul Retain Set ---")

unlearned_model, best_val_p10 = uf.run_finetune_with_p10(
    model=unlearned_model,
    train_loader=train_loader,
    X_val=X_val,
    y_val=y_val,
    device=device,
    uf_module=uf,
    epochs=2,
    lr=3e-4
)

submission_path = uf.export_submission(
    model=unlearned_model,
    val_df=val_df,
    id_col=id_col,
    architecture=architecture,
    best_params=best_params,
    payload=payload,
    version="V18",
    group_name="NoWINDtoday"
)


--- FASE 1: Unlearning con KL Divergence ---
Unlearning con KL completato!

--- FASE 2: Fine-Tuning di Ricalibrazione sul Retain Set ---

--- Inizio Fine-Tuning sul Retain Set (2 epoche) ---
Epoch 1/2 | Retain Loss: 0.1229 | Train P@10: 0.0355 | Val P@10: 0.0402
Epoch 2/2 | Retain Loss: 0.0817 | Train P@10: 0.0410 | Val P@10: 0.0411
✅ Caricati i pesi dell'epoca con la migliore Val Precision@10: 0.0411
--- Check Dtypes dello State Dict ---
Tensori Float32: 14 | Tensori Float64: 0
✅ Perfetto: NESSUN tensore a 64-bit presente! Tutto in Float32.

✅ Cartella 'submissions\NoWINDtoday_V18' creata con successo e pronta per l'invio!


# Influence

In [7]:
import time

start_time = time.time()

# 1. Esegui l'Unlearning basato sulle Influence Functions
influence_model = uf.influence_unlearning(
    model=model,
    forget_loader=forget_loader,
    device=device,
    lr=1e-3,       # Se distrugge troppo abbassa a 1e-4
    damping=1e-2,  # Stabilizzatore del gradiente
    scale=5.0      # Intensità dell'amnesia
)

# 2. Fine-Tuning rapido sul Retain Set per ricalibrare il Ranking (utilizza la funzione creata prima!)
final_influence_model, best_val_p10 = uf.run_finetune_with_p10(
    model=influence_model,
    train_loader=train_loader,
    X_val=X_val,
    y_val=y_val,
    device=device,
    uf_module=uf,
    epochs=2,
    lr=5e-4
)

# 3. Creazione della Submission Blindata con export_submission
submission_path = uf.export_submission(
    model=final_influence_model,
    val_df=val_df,
    id_col=id_col,
    architecture=architecture,
    best_params=best_params,
    version="V32",
    group_name="NoWINDtoday",
    payload=payload,
)


--- Inizio Influence-based Unlearning ---
✅ Influence-based Unlearning completato con successo!

--- Inizio Fine-Tuning sul Retain Set (2 epoche) ---
Epoch 1/2 | Retain Loss: 0.1054 | Train P@10: 0.0374 | Val P@10: 0.0410
Epoch 2/2 | Retain Loss: 0.0803 | Train P@10: 0.0415 | Val P@10: 0.0411
✅ Caricati i pesi dell'epoca con la migliore Val Precision@10: 0.0411
--- Check Dtypes dello State Dict ---
Tensori Float32: 14 | Tensori Float64: 0
✅ Perfetto: NESSUN tensore a 64-bit presente! Tutto in Float32.

✅ Cartella 'submissions\NoWINDtoday_V32' creata con successo e pronta per l'invio!


# SSD

In [6]:
import time

start_time = time.time()

# 1. Esegui l'SSD Unlearning
ssd_unlearned_model = uf.ssd_unlearning(
    model=model,
    forget_loader=forget_loader,
    retain_loader=train_loader,
    device=device,
    dampening_constant=1.0,  # alpha: aumenta se vuoi cancellare più aggressivamente
    selection_threshold=1.0  # lambda: abbassa per selezionare più pesi
)

# 2. Fine-Tuning lampo di ricalibrazione (2 epoche)
final_ssd_model, best_val_p10 = uf.run_finetune_with_p10(
    model=ssd_unlearned_model,
    train_loader=train_loader,
    X_val=X_val,
    y_val=y_val,
    device=device,
    uf_module=uf,
    epochs=2,
    lr=5e-4
)

# 3. Creazione della Submission Blindata
submission_path = uf.export_submission(
    model=final_ssd_model,
    val_df=val_df,
    id_col=id_col,
    architecture=architecture,
    best_params=best_params,
    version="V30",
    group_name="NoWINDtoday",
    payload=payload,
    start_time=start_time
)


--- Inizio Selective Synaptic Dampening (SSD) Unlearning ---
✅ SSD Unlearning completato con successo!

--- Inizio Fine-Tuning sul Retain Set (2 epoche) ---
Epoch 1/2 | Retain Loss: 0.1065 | Train P@10: 0.0373 | Val P@10: 0.0409
Epoch 2/2 | Retain Loss: 0.0805 | Train P@10: 0.0415 | Val P@10: 0.0411
✅ Caricati i pesi dell'epoca con la migliore Val Precision@10: 0.0411
--- Check Dtypes dello State Dict ---
Tensori Float32: 14 | Tensori Float64: 0
✅ Perfetto: NESSUN tensore a 64-bit presente! Tutto in Float32.

✅ Cartella 'submissions\NoWINDtoday_V30' creata con successo e pronta per l'invio!


# Retraining from scratch

In [15]:
import os
import time
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.impute import SimpleImputer
from torch.utils.data import DataLoader, TensorDataset


# 0. Impostazione del Random Seed per riproducibilità
random_seed = 456
np.random.seed(random_seed)
torch.manual_seed(random_seed)

# 1. Isolamento completo del Retain Set e pulizia imputer (ZERO Data Leakage)
X_retain_raw, y_retain, feature_cols, target_cols = uf.prepare_data(
    df_all, id_col=id_col, target_prefix='target__'
)

imputer_scratch = SimpleImputer(strategy='median')
X_retain_scratch = imputer_scratch.fit_transform(X_retain_raw).astype(np.float32)

# Imputazione e preparazione per il Validation Set
X_val_raw, y_val, _, _ = uf.prepare_data(
    val_df, id_col=id_col, target_prefix='target__'
)
X_val_scratch = imputer_scratch.transform(X_val_raw).astype(np.float32)

# 2. Creazione dei DataLoader (Train & Validation)
batch_size = best_params.get('batch_size', 256) if 'best_params' in locals() else 256

retain_dataset = TensorDataset(
    torch.as_tensor(X_retain_scratch, dtype=torch.float32),
    torch.as_tensor(y_retain, dtype=torch.float32)
)
retain_loader = DataLoader(retain_dataset, batch_size=batch_size, shuffle=True)

val_dataset = TensorDataset(
    torch.as_tensor(X_val_scratch, dtype=torch.float32),
    torch.as_tensor(y_val, dtype=torch.float32)
)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# 3. Loss Function per il Ranking (BCE Standard per massimizzare la Precision@10)
criterion = nn.BCEWithLogitsLoss().to(device)

# 4. Inizializzazione della rete EX NOVO
retrained_model = DynamicMLP(
    input_dim=architecture['input_dim'],
    hidden_layers=architecture['hidden_layers'],
    num_outputs=architecture['num_outputs']
).to(device)

start_time = time.time()


lr = 0.001
epochs = 5
optimizer = torch.optim.Adam(retrained_model.parameters(), lr=lr)

print(f"\n--- Inizio Retraining da Zero ({epochs} epoche) con Monitoraggio Precision@10 ---")

best_p10 = -1.0
best_model_weights = None

for epoch in range(epochs):
    # --- PASSO DI TRAINING ---
    retrained_model.train()
    total_loss = 0.0
    train_p10_list = []

    for x_b, y_b in retain_loader:
        x_b, y_b = x_b.to(device), y_b.to(device)

        optimizer.zero_grad()
        out = retrained_model(x_b)
        loss = criterion(out, y_b)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # Calcolo Precision@10 in addestramento
        with torch.no_grad():
            batch_p10 = uf.precision_at_k(out, y_b, k=10)
            train_p10_list.append(batch_p10)

    mean_loss = total_loss / len(retain_loader)
    mean_train_p10 = sum(train_p10_list) / len(train_p10_list)

    # --- PASSO DI VALIDAZIONE ---
    retrained_model.eval()
    val_p10_list = []

    with torch.no_grad():
        for x_v, y_v in val_loader:
            x_v, y_v = x_v.to(device), y_v.to(device)
            out_v = retrained_model(x_v)
            
            p10_v = uf.precision_at_k(out_v, y_v, k=10)
            val_p10_list.append(p10_v)

    mean_val_p10 = sum(val_p10_list) / len(val_p10_list)

    # Checkpoint sui pesi con la Val Precision@10 più alta
    if mean_val_p10 > best_p10:
        best_p10 = mean_val_p10
        best_model_weights = copy.deepcopy(retrained_model.state_dict())

    # Logging programmatico ogni 5 epoche o all'ultima
    if (epoch + 1) % 5 == 0 or epoch == epochs - 1:
        print(
            f"Epoch {epoch+1:02d}/{epochs:02d} | "
            f"Train Loss: {mean_loss:.4f} | "
            f"Train P@10: {mean_train_p10:.4f} | "
            f"Val P@10: {mean_val_p10:.4f}"
        )

# Ripristino dello stato con la Precision@10 migliore ottenuta
if best_model_weights is not None:
    retrained_model.load_state_dict(best_model_weights)
    print(f"\n🎯 Caricati i pesi dell'epoca con il RECORD di Val Precision@10: {best_p10:.4f}")

retrained_model.eval()
print("Retraining da zero completato!")


# =========================================================
# CREAZIONE SUBMISSION BLINDATA (Uso di export_submission)
# =========================================================
submission_path = uf.export_submission(
    model=retrained_model,
    val_df=val_df,
    id_col=id_col,
    architecture=architecture,
    best_params={'lr': lr, 'epochs': epochs, 'batch_size': batch_size},
    version="V36",
    group_name="NoWINDtoday",
    payload=payload
)

DataFrame columns: ['ang_m_datediff_att_linea', 'ang_m_flag_mnp', 'ang_m_datediff_cessazione', 'ang_m_datediff_scad_sim', 'ang_m_cns_cre_res_per', 'ang_m_anz_linea', 'ang_m_datediff_mnp_in', 'ang_m_datediff_cam_prof', 'ang_m_datediff_mnp_out', 'ohe_ang_m_stato_linea_a', 'ohe_ang_m_stato_linea_c', 'ohe_ang_m_stato_linea_n', 'ohe_ang_m_stato_linea_s', 'ohe_ang_m_tipo_linea_fwa', 'ohe_ang_m_tipo_linea_pp', 'ohe_ang_m_tipo_linea_xt', 'ohe_ang_m_area_territoriale_altro_nd', 'ohe_ang_m_area_territoriale_c', 'ohe_ang_m_area_territoriale_ne', 'ohe_ang_m_area_territoriale_no', 'ohe_ang_m_area_territoriale_s', 'ohe_ang_m_pdv_canale_des_top_4_altro', 'ohe_ang_m_pdv_canale_des_top_4_grande_distribuzione', 'ohe_ang_m_pdv_canale_des_top_4_null', 'ohe_ang_m_pdv_canale_des_top_4_one_team', 'ohe_ang_m_pdv_canale_des_top_4_retail', 'ohe_ang_m_pdv_canale_des_top_4_tele_sales', 'ohe_ang_m_mnp_olo_prov_infrastrutturati', 'ohe_ang_m_mnp_olo_prov_mvno', 'ohe_ang_m_mnp_olo_prov_nd', 'ohe_ang_m_mnp_olo_dest_in